# Demo: Fine-tuning BERT, GPT-2, and T5

## 1. BERT for sentiment classification

- Why BERT?: strong at classification with minimal training
- Task: sentence-level sentiment classification
- Dataset (Hugging Face datasets names): SST-2 (GLUE) – glue

## 2. GPT-2 for dialogue/response generation

- Why GPT?: natural for next-token generation
- Task: short-turn response generation
- Dataset: DailyDialog – daily_dialog

## 3. T5 for dialogue summarization

- Why T5: designed for seq2seq; single text-to-text interface
- Task: summarization (text-to-text)
- Dataset: SAMSum – samsum

# Setup (uv)

```bash
uv venv
source .venv/bin/activate   # Windows: .venv\Scripts\activate
uv sync                     # resolves deps, writes uv.lock
uv run jupyter notebook
```

# Pre-download all datasets

(Optional, or each notebook will also download its dataset at the first cell)

```bash
uv run python - << 'PY'
from datasets import load_dataset
print('Downloading SST-2 ...'); load_dataset('glue','sst2')
print('Downloading DailyDialog ...'); load_dataset('daily_dialog')
print('Downloading SAMSum ...'); load_dataset('samsum')
print('Done.')
PY
```

In [ ]:
import os, json, random

import torch
import numpy as np
from datasets import load_dataset
from rouge_score import rouge_scorer
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Trainer, TrainingArguments,
    DataCollatorForSeq2Seq
)

In [ ]:
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Pre-download SAMSum
raw_data = load_dataset('samsum')
print('SAMSum done downloading.')

In [ ]:
model_name = 't5-small'
output_dir = 'outputs/t5-small-samsum'

In [ ]:
# Tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(DEVICE)
collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
# Tokenize and preprocess data
prefix = 'summarize: '
input_max_length = 512
target_max_length = 128

def preprocess(batch):
    inputs = [prefix + d for d in batch['dialogue']]
    model_inputs = tokenizer(inputs, max_length=input_max_length, truncation=True)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(batch['summary'], max_length=target_max_length, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_data = raw_data.map(preprocess, batched=True, remove_columns=raw_data['train'].column_names)

In [ ]:
# ROUGE metrics
scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple): preds = preds[0]
    decoded_preds = tok.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tok.pad_token_id)
    decoded_labels = tok.batch_decode(labels, skip_special_tokens=True)
    scores = {k: [] for k in ['rouge1','rouge2','rougeL']}
    for p, l in zip(decoded_preds, decoded_labels):
        s = scorer.score(l, p)
        for k in scores: scores[k].append(s[k].fmeasure)
    return {k: float(np.mean(v)) for k, v in scores.items()}

In [ ]:
# Train
args = TrainingArguments(
    output_dir=output_dir,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=target_max_length,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to=[],
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_data['train'],
    eval_dataset=tokenized_data['validation'],
    data_collator=collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)
trainer.train()

In [ ]:
os.makedirs(output_dir, exist_ok=True)
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

In [ ]:
metrics = trainer.evaluate()
metrics

In [ ]:
with open(os.path.join(output_dir, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved to', output_dir, '', metrics)

In [ ]:
# Quick summarization
text = """
    Tom: Hey, are we still on for 5pm?
    Anna: Running 10 minutes late!
    Tom: No worries, see you soon.
"""

In [ ]:
model.eval()

In [ ]:
enc = tokenizer('summarize: ' + text, return_tensors='pt').to(DEVICE)

In [ ]:
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=128)       

In [ ]:
text

In [ ]:
tokenizer.decode(out[0], skip_special_tokens=True)